In [4]:
import numpy as np
import cv2
import pickle
import os

# Load Phase 2 results
with open('phase2_results.pkl', 'rb') as f:
    p2 = pickle.load(f)

img = p2['img']
height, width = p2['height'], p2['width']
region_map = p2['region_map']
special_regions = p2['special_regions']

output_dir = "comic/phase3_output"
os.makedirs(output_dir, exist_ok=True)

print(f"Phase 2 data loaded")
print(f"Special regions (text+edge): {len(special_regions)}")

# Get gradient candidate regions (not special)
gradient_candidates = set(np.unique(region_map[region_map >= 0])) - special_regions
print(f"Gradient candidate regions: {len(gradient_candidates)}")

Phase 2 data loaded
Special regions (text+edge): 11387
Gradient candidate regions: 2271


In [5]:
# Precompute Sobel responses once for the whole image
# (the old version recomputed them for every region)
gray_f = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(float)
Gx = cv2.Sobel(gray_f, cv2.CV_64F, 1, 0, ksize=3)
Gy = cv2.Sobel(gray_f, cv2.CV_64F, 0, 1, ksize=3)
grad_mag = np.sqrt(Gx**2 + Gy**2)
grad_theta = np.arctan2(Gy, Gx)

def gradient_endpoints(pixels_yx, direction, img, frac=0.05):
    """
    Project pixels onto the gradient direction and return the mean colour
    of the lowest and highest `frac` of the projection (start/end colours).
    Pixels are converted to (x, y) order so the direction vector matches
    the Sobel angle convention (the old code mixed up x and y).
    """
    xy = pixels_yx[:, ::-1].astype(float)
    angle = np.radians(direction * 45)
    proj = xy @ np.array([np.cos(angle), np.sin(angle)])
    order = np.argsort(proj)
    k = max(1, int(len(order) * frac))
    lo = pixels_yx[order[:k]]
    hi = pixels_yx[order[-k:]]
    start = img[lo[:, 0], lo[:, 1]].astype(float).mean(axis=0)
    end = img[hi[:, 0], hi[:, 1]].astype(float).mean(axis=0)
    return start, end

def detect_gradient(rid, rmap, img, mag_threshold=10):
    region_mask = (rmap == rid)
    pixels_yx = np.argwhere(region_mask)
    if len(pixels_yx) < 10:
        return 'flat'
    if np.mean(grad_mag[region_mask]) < mag_threshold:
        return 'flat'

    theta_deg = (np.degrees(grad_theta[region_mask]) + 180) % 360
    direction_bin = (theta_deg / 45).astype(int) % 8
    hist, _ = np.histogram(direction_bin, bins=np.arange(0, 9))
    if np.max(hist) < len(pixels_yx) * 0.3:
        return 'flat'

    main_direction = int(np.argmax(hist))
    start_color, end_color = gradient_endpoints(pixels_yx, main_direction, img)
    return ('gradient', main_direction, start_color, end_color)

# Classify regions
gradient_regions = {}
flat_regions = []

for rid in gradient_candidates:
    result = detect_gradient(rid, region_map, img)
    if result == 'flat':
        flat_regions.append(rid)
    else:
        gradient_regions[rid] = {
            'direction': result[1],
            'start_color': result[2],
            'end_color': result[3]
        }

print(f"\nGradient detection results:")
print(f"  Gradient regions: {len(gradient_regions)}")
print(f"  Flat regions: {len(flat_regions)}")


Gradient detection results:
  Gradient regions: 315
  Flat regions: 1956


In [6]:
def get_direct_neighbors(rid, rmap):
    mask = (rmap == rid).astype(np.uint8)
    # A 3x3 cross dilation does not wrap around the image borders (np.roll did)
    kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))
    ring = cv2.dilate(mask, kernel).astype(bool) & ~mask.astype(bool)
    ids = np.unique(rmap[ring])
    return [int(i) for i in ids if i != rid and i != -1]

def directions_compatible(dir1, dir2, tolerance=1):
    diff = abs(dir1 - dir2)
    diff = min(diff, 8 - diff)
    return diff <= tolerance

# Union-Find (reset on every run so stale entries cannot leak in)
parent = {}

def find(x):
    if x not in parent:
        parent[x] = x
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    px, py = find(x), find(y)
    if px != py:
        parent[px] = py

# Find adjacent gradient regions with compatible directions
for rid in gradient_regions.keys():
    for nid in get_direct_neighbors(rid, region_map):
        if nid in gradient_regions and directions_compatible(
                gradient_regions[rid]['direction'], gradient_regions[nid]['direction']):
            union(rid, nid)

groups = {}
for rid in gradient_regions.keys():
    groups.setdefault(find(rid), []).append(rid)

print(f"After merging adjacent gradients: {len(groups)} groups")

# Build one merged gradient region per group
merged_gradient_regions = {}

for root, members in groups.items():
    group_mask = np.isin(region_map, members)
    pixels_yx = np.argwhere(group_mask)
    direction = gradient_regions[root]['direction']
    start_color, end_color = gradient_endpoints(pixels_yx, direction, img)

    # Relabel every member with the root id, so later phases see ONE region.
    # Without this, the non-root members were silently demoted to flat regions.
    region_map[group_mask] = root

    merged_gradient_regions[root] = {
        'direction': direction,
        'start_color': start_color,
        'end_color': end_color,
        'merged_from': members
    }

gradient_regions = merged_gradient_regions
print(f"Merged gradient regions: {len(gradient_regions)}")

After merging adjacent gradients: 226 groups
Merged gradient regions: 226


In [7]:
phase3_results = {
    'img': img,
    'height': height,
    'width': width,
    'region_map': region_map,
    'special_regions': special_regions,
    'gradient_regions': gradient_regions,
    'flat_regions': flat_regions
}

with open('phase3_results.pkl', 'wb') as f:
    pickle.dump(phase3_results, f)

print("Phase 3 results saved to phase3_results.pkl")
print(f"Summary:")
print(f"  Special regions (text+edge): {len(special_regions)}")
print(f"  Gradient regions: {len(gradient_regions)}")
print(f"  Flat regions: {len(flat_regions)}")

Phase 3 results saved to phase3_results.pkl
Summary:
  Special regions (text+edge): 11387
  Gradient regions: 226
  Flat regions: 1956


In [8]:
# Visualize the classification results

# Create visualization
vis = img.copy().astype(float)

# Mark special regions (text+edge) - Red
for rid in special_regions:
    mask = (region_map == rid)
    vis[mask] = [255, 0, 0]  # Red

# Mark gradient regions - Green
for rid in gradient_regions.keys():
    mask = (region_map == rid)
    vis[mask] = [0, 255, 0]  # Green

# Mark flat regions - Blue (just a few for reference)
flat_count = 0
for rid in flat_regions:
    if flat_count < 100:  # Just show first 100 for visibility
        mask = (region_map == rid)
        vis[mask] = [0, 0, 255]  # Blue
        flat_count += 1

# Save visualization
vis_path = os.path.join(output_dir, "01_phase3_classification.jpg")
cv2.imwrite(vis_path, cv2.cvtColor(vis.astype(np.uint8), cv2.COLOR_RGB2BGR))
print(f"Classification visualization saved: {vis_path}")

print(f"\nColor legend:")
print(f"  Red = Special regions (text+edge)")
print(f"  Green = Gradient regions")
print(f"  Blue = Flat regions (sample)")

Classification visualization saved: comic/phase3_output\01_phase3_classification.jpg

Color legend:
  Red = Special regions (text+edge)
  Green = Gradient regions
  Blue = Flat regions (sample)


In [9]:
import numpy as np

def find_gradient_seams(region_map, gradient_regions, img, jump_threshold=40):
    """
    Scan for abrupt color jumps within and between gradient regions.
    A properly reconstructed gradient should have small pixel-to-pixel
    color differences; a seam/discontinuity shows up as a large jump
    at region boundaries.
    """
    seams_found = []
    grad_rids = set(gradient_regions.keys())

    height, width = region_map.shape

    # Check horizontal neighbors
    for y in range(height):
        for x in range(width - 1):
            rid1 = region_map[y, x]
            rid2 = region_map[y, x + 1]
            if rid1 == rid2:
                continue
            if rid1 not in grad_rids and rid2 not in grad_rids:
                continue
            # only check where at least one side is a gradient region
            c1 = img[y, x].astype(float)
            c2 = img[y, x + 1].astype(float)
            diff = np.sqrt(np.sum((c1 - c2) ** 2))
            if diff > jump_threshold:
                seams_found.append((y, x, int(rid1), int(rid2), diff))

    return seams_found

seams = find_gradient_seams(region_map, gradient_regions, img)
print(f"Total seam points found: {len(seams)}")

if seams:
    # Show a few examples with the biggest jumps
    seams_sorted = sorted(seams, key=lambda s: -s[4])
    print("\nTop 10 largest jumps:")
    for y, x, r1, r2, diff in seams_sorted[:10]:
        print(f"  at ({y},{x}): region {r1} -> region {r2}, color jump={diff:.1f}")

    # Check: are the two regions in the SAME merged group, or different groups?
    # If same group but still jumping, the merge/interpolation logic is broken.
    # If different groups, the issue is groups that should have merged didn't.
    print("\nChecking if biggest-jump pairs belong to the same merged group:")
    for y, x, r1, r2, diff in seams_sorted[:5]:
        in_r1 = r1 in gradient_regions
        in_r2 = r2 in gradient_regions
        dir1 = gradient_regions[r1]['direction'] if in_r1 else None
        dir2 = gradient_regions[r2]['direction'] if in_r2 else None
        print(f"  ({y},{x}): r1={r1}(dir={dir1}) r2={r2}(dir={dir2}) jump={diff:.1f}")

Total seam points found: 12219

Top 10 largest jumps:
  at (62,391): region 5849 -> region 38216, color jump=398.4
  at (61,351): region 5420 -> region 38216, color jump=394.3
  at (46,455): region 38216 -> region 1644, color jump=391.4
  at (39,688): region 1709 -> region 889, color jump=386.2
  at (94,703): region 8800 -> region 889, color jump=382.8
  at (265,751): region 24876 -> region 25583, color jump=382.8
  at (267,655): region 38216 -> region 25680, color jump=382.8
  at (229,767): region 38216 -> region 20069, color jump=381.1
  at (236,687): region 38216 -> region 20895, color jump=381.1
  at (266,743): region 38216 -> region 25450, color jump=381.1

Checking if biggest-jump pairs belong to the same merged group:
  (62,391): r1=5849(dir=None) r2=38216(dir=6) jump=398.4
  (61,351): r1=5420(dir=None) r2=38216(dir=6) jump=394.3
  (46,455): r1=38216(dir=6) r2=1644(dir=None) jump=391.4
  (39,688): r1=1709(dir=None) r2=889(dir=5) jump=386.2
  (94,703): r1=8800(dir=None) r2=889(di